In [ ]:
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data" / "processed").exists():
    REPO_ROOT = Path.cwd().parent

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
FIGURES_DIR = REPO_ROOT / "results" / "figures" / "preprocessed"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

METADATA_FILE = PROCESSED_DIR / "metadata_filtered.parquet"
EXPRESSION_FILE = PROCESSED_DIR / "expression_filtered.parquet"
GENE_LIST_FILE = PROCESSED_DIR / "gene_list.txt"
GENE_STATS_FILE = PROCESSED_DIR / "gene_stats.csv"

ENSG_PATTERN = re.compile(r"^ENSG\d+$")

PALETTE = {
    "primary": "#4C6EF5", "secondary": "#74C0FC",
    "accent":  "#F76707", "green":    "#51CF66",
    "purple":  "#845EF7", "pink":     "#F06595",
    "teal":    "#20C997", "gray":     "#ADB5BD",
    "bg":      "#F8F9FA", "dark":     "#212529",
}

# MoE domain colours (high contrast; not blue vs purple)
DOMAIN = {
    "neural": "#D95F02",
    "non_neural": "#1B9E77",
    "other": "#7A7A7A",
}
NEURAL_KEYWORDS = ["brain", "spinal cord", "nerve"]

def is_neural_tissue(t):
    name = str(t).lower()
    return any(k in name for k in NEURAL_KEYWORDS)

def tissue_domain_color(t):
    if str(t).startswith("Other"):
        return DOMAIN["other"]
    return DOMAIN["neural"] if is_neural_tissue(t) else DOMAIN["non_neural"]

plt.rcParams.update({
    "figure.facecolor": PALETTE["bg"],
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.titlesize":   13,
    "axes.titleweight": "bold",
    "grid.color":       "#E9ECEF",
    "grid.linewidth":   0.8,
})
print(f"Repo root: {REPO_ROOT}")
print("Ready: Processed data EDA initialized.")

Repo root: c:\Research\Code\gtex-bioage
Ready: Processed data EDA initialized.


In [13]:
# Load processed metadata (already filtered to match expression samples)
# SAMPID is index, AGE_MID already numeric, 8 columns total
meta = pd.read_parquet(METADATA_FILE)

print(f"Processed metadata shape: {meta.shape}")
print(f"Columns: {list(meta.columns)}")
print(f"SAMPID is index: {meta.index.name == 'SAMPID'}")
print(f"Unique subjects: {meta['SUBJID'].nunique():,}")
print(f"Unique tissues: {meta['SMTSD'].nunique()}")
print(f"AGE_MID range: {meta['AGE_MID'].min():.0f} - {meta['AGE_MID'].max():.0f}")
print(f"\nFirst 2 rows:")
print(meta.head(2))

Processed metadata shape: (19675, 8)
Columns: ['SUBJID', 'SMTSD', 'SMTS', 'AGE', 'SEX', 'DTHHRDY', 'SMRIN', 'AGE_MID']
SAMPID is index: True
Unique subjects: 930
Unique tissues: 68
AGE_MID range: 25 - 75

First 2 rows:
                                   SUBJID                         SMTSD  \
SAMPID                                                                    
GTEX-1117F-0005-SM-HL9SH       GTEX-1117F                   Whole Blood   
GTEX-1117F-0011-R10b-SM-GI4VE  GTEX-1117F  Brain - Frontal Cortex (BA9)   

                                SMTS    AGE SEX  DTHHRDY  SMRIN  AGE_MID  
SAMPID                                                                    
GTEX-1117F-0005-SM-HL9SH       Blood  60-69   F      4.0    8.3     65.0  
GTEX-1117F-0011-R10b-SM-GI4VE  Brain  60-69   F      4.0    7.2     65.0  


In [14]:
# Validate ENSG gene IDs and inspect expression matrix (samples x genes)
gene_list = Path(GENE_LIST_FILE).read_text().strip().split("\n")
assert len(gene_list) == 5000, f"Expected 5000 genes, got {len(gene_list)}"
bad_genes = [g for g in gene_list if not ENSG_PATTERN.match(g)]
assert not bad_genes, f"Non-ENSG IDs in gene_list.txt: {bad_genes[:3]}"
print(f"ENSG validation OK: {len(gene_list):,} genes")
print(f"Examples: {gene_list[0]}, {gene_list[1]}, {gene_list[2]}")

pf = pq.ParquetFile(EXPRESSION_FILE)
schema_cols = pf.schema_arrow.names
gene_cols = [c for c in schema_cols if ENSG_PATTERN.match(c)]
n_expr_samples = pd.read_parquet(EXPRESSION_FILE, columns=[]).shape[0]

print(f"\nExpression parquet row groups: {pf.metadata.num_row_groups}")
print(f"Samples (rows): {n_expr_samples:,}")
print(f"Gene columns (ENSG): {len(gene_cols):,}")
print(f"First 3 gene columns: {gene_cols[:3]}")

expr_head = pd.read_parquet(EXPRESSION_FILE).head(3)
print(f"Sample index name: {expr_head.index.name}")
print(f"First 2 sample IDs: {expr_head.index[:2].tolist()}")

assert set(gene_list) == set(gene_cols), "gene_list.txt and expression columns differ"
assert meta.index.equals(pd.read_parquet(EXPRESSION_FILE, columns=[]).index), "Metadata/expression sample mismatch"
print("Sample alignment: OK")

ENSG validation OK: 5,000 genes
Examples: ENSG00000175084, ENSG00000163017, ENSG00000211592

Expression parquet row groups: 40
Samples (rows): 19,675
Gene columns (ENSG): 5,000
First 3 gene columns: ['ENSG00000308579', 'ENSG00000269981', 'ENSG00000225972']
Sample index name: SAMPID
First 2 sample IDs: ['GTEX-1117F-0005-SM-HL9SH', 'GTEX-1117F-0011-R10b-SM-GI4VE']
Sample alignment: OK


In [15]:
n_subjects = meta["SUBJID"].nunique()
n_samples = meta.shape[0]
n_tissues = meta["SMTSD"].nunique()
n_genes = len(gene_list)
n_expr_samples = n_samples

print(f"""
╔══════════════════════════════════════════════════╗
║         GTEx v11 Processed Dataset Summary       ║
╠══════════════════════════════════════════════════╣
║  Expression Matrix (samples x ENSG genes)        ║
║    Samples (rows)         : {n_samples:>10,}           ║
║    Genes (columns)        : {n_genes:>10,}           ║
║                                                  ║
║  Filtered Metadata                               ║
║    Total subjects       : {n_subjects:>10,}             ║
║    Total samples        : {n_samples:>10,}             ║
║    Tissue types         : {n_tissues:>10}             ║
╚══════════════════════════════════════════════════╝
""")

print("AGE_MID (numeric age) distribution:")
print(meta["AGE_MID"].describe())
print("\nSex distribution (M/F):")
print(meta["SEX"].value_counts())
print("\nHardy Scale:")
hardy_map = {0:"Ventilator", 1:"Violent/Fast", 2:"Fast illness",
             3:"Intermediate", 4:"Slow illness"}
print(meta["DTHHRDY"].map(hardy_map).value_counts().sort_index())


╔══════════════════════════════════════════════════╗
║         GTEx v11 Processed Dataset Summary       ║
╠══════════════════════════════════════════════════╣
║  Expression Matrix (samples x ENSG genes)        ║
║    Samples (rows)         :     19,675           ║
║    Genes (columns)        :      5,000           ║
║                                                  ║
║  Filtered Metadata                               ║
║    Total subjects       :        930             ║
║    Total samples        :     19,675             ║
║    Tissue types         :         68             ║
╚══════════════════════════════════════════════════╝

AGE_MID (numeric age) distribution:
count    19675.000000
mean        53.678526
std         12.738807
min         25.000000
25%         45.000000
50%         55.000000
75%         65.000000
max         75.000000
Name: AGE_MID, dtype: float64

Sex distribution (M/F):
SEX
M    13196
F     6479
Name: count, dtype: int64

Hardy Scale:
DTHHRDY
Fast illness    5516


In [16]:
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(18, 11), facecolor=PALETTE["bg"])
fig.suptitle("GTEx v11 Processed Dataset Overview",
             fontsize=18, fontweight="bold", color=PALETTE["dark"], y=1.01)
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Sex donut (M/F)
ax1 = fig.add_subplot(gs[0, 0])
sex_counts = meta["SEX"].value_counts()
ax1.pie(sex_counts, labels=sex_counts.index,
        colors=[PALETTE["primary"], PALETTE["pink"]],
        autopct="%1.1f%%", startangle=90,
        wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2))
ax1.set_title("Sex Distribution (Processed)")
ax1.text(0, 0, f"{sex_counts.sum():,}\nsamples",
         ha="center", va="center", fontsize=9, fontweight="bold")

# Age bar (decade bins)
ax2 = fig.add_subplot(gs[0, 1])
age_counts = meta["AGE"].value_counts().sort_index()
colors_age = plt.cm.viridis(np.linspace(0.2, 0.9, len(age_counts)))
bars = ax2.bar(range(len(age_counts)), age_counts.values,
               color=colors_age, edgecolor="white", linewidth=1.5, width=0.7)
ax2.set_xticks(range(len(age_counts)))
ax2.set_xticklabels(age_counts.index, rotation=30)
ax2.set_ylabel("Samples")
ax2.set_title("Age Group Distribution")
ax2.yaxis.grid(True, linestyle="--")
ax2.set_axisbelow(True)
for b in bars:
    ax2.text(b.get_x() + b.get_width() / 2, b.get_height() + 50,
             f"{int(b.get_height()):,}", ha="center", fontsize=8)

# Hardy Scale
ax3 = fig.add_subplot(gs[0, 2])
hardy_map = {0:"Ventilator", 1:"Violent/Fast", 2:"Fast illness",
             3:"Intermediate", 4:"Slow illness"}
hc = meta["DTHHRDY"].map(hardy_map).value_counts().sort_index()
hcolors = [PALETTE["green"], PALETTE["teal"], PALETTE["secondary"],
           PALETTE["accent"], PALETTE["gray"]]
bars = ax3.barh(range(len(hc)), hc.values,
                color=hcolors[:len(hc)], edgecolor="white", height=0.6)
ax3.set_yticks(range(len(hc)))
ax3.set_yticklabels(hc.index, fontsize=9)
ax3.set_xlabel("Samples")
ax3.set_title("Hardy Scale\n(Death Circumstances)")
ax3.xaxis.grid(True, linestyle="--")
ax3.set_axisbelow(True)
for b in bars:
    ax3.text(b.get_width() + 50, b.get_y() + b.get_height() / 2,
             f"{int(b.get_width()):,}", va="center", fontsize=8)

# Top 20 tissues (neural vs non-neural colours)
ax4 = fig.add_subplot(gs[1, :2])
tc = meta["SMTSD"].value_counts().head(20).sort_values(ascending=True)
cols_t = [tissue_domain_color(t) for t in tc.index]
bars = ax4.barh(range(len(tc)), tc.values,
                color=cols_t, edgecolor="white", height=0.7)
ax4.set_yticks(range(len(tc)))
ax4.set_yticklabels(tc.index, fontsize=8)
ax4.set_xlabel("Samples")
ax4.set_title("Top 20 Tissues by Sample Count")
ax4.xaxis.grid(True, linestyle="--")
ax4.set_axisbelow(True)
for b in bars:
    ax4.text(b.get_width() + 2, b.get_y() + b.get_height() / 2,
             f"{int(b.get_width()):,}", va="center", fontsize=7.5)

# RIN quality
ax5 = fig.add_subplot(gs[1, 2])
rin = meta["SMRIN"].dropna()
ax5.hist(rin, bins=30, color=PALETTE["teal"], edgecolor="white", alpha=0.9)
ax5.axvline(rin.median(), color=PALETTE["accent"], linestyle="--", linewidth=2,
            label=f"Median: {rin.median():.1f}")
ax5.set_xlabel("RNA Integrity Number (RIN)")
ax5.set_ylabel("Samples")
ax5.set_title("RNA Quality (SMRIN)")
ax5.legend(fontsize=9)
ax5.yaxis.grid(True, linestyle="--")
ax5.set_axisbelow(True)

fig.set_constrained_layout(True)
plt.savefig(FIGURES_DIR / "01_overview_dashboard.png",
            dpi=150, bbox_inches="tight", facecolor=PALETTE["bg"])
plt.close()
print("Saved -> preprocessed/01_overview_dashboard.png")

Saved -> preprocessed/01_overview_dashboard.png


In [17]:
tissue_counts = meta['SMTSD'].value_counts().sort_values(ascending=True)
colors = [tissue_domain_color(t) for t in tissue_counts.index]

fig, ax = plt.subplots(figsize=(12, max(12, len(tissue_counts) * 0.28)),
                       facecolor=PALETTE["bg"])
fig.suptitle("GTEx v11 Processed — All Tissues Sample Distribution",
             fontsize=14, fontweight="bold", color=PALETTE["dark"])

bars = ax.barh(range(len(tissue_counts)), tissue_counts.values,
               color=colors, edgecolor="white", linewidth=0.5, height=0.75)
ax.set_yticks(range(len(tissue_counts)))
ax.set_yticklabels(tissue_counts.index, fontsize=7.5)
ax.set_xlabel("Number of Samples")
ax.xaxis.grid(True, linestyle="--", alpha=0.7)
ax.set_axisbelow(True)

for b in bars:
    ax.text(b.get_width() + 1, b.get_y() + b.get_height() / 2,
            f"{int(b.get_width()):,}", va="center", fontsize=6.5)

neural_p = mpatches.Patch(color=DOMAIN["neural"], label="Neural tissues")
other_p  = mpatches.Patch(color=DOMAIN["non_neural"], label="Non-neural tissues")
ax.legend(handles=[neural_p, other_p], loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_tissue_distribution.png",
            dpi=150, bbox_inches="tight", facecolor=PALETTE["bg"])
plt.close()
print("Saved -> preprocessed/02_tissue_distribution.png")

Saved -> preprocessed/02_tissue_distribution.png


In [ ]:
# Top 20 tissues + Other (thesis-friendly summary chart)
TOP_N = 20
tissue_counts = meta["SMTSD"].value_counts()
top = tissue_counts.head(TOP_N)
other_n = int(tissue_counts.iloc[TOP_N:].sum())
other_label = f"Other ({len(tissue_counts) - TOP_N} tissues)"
summary = pd.concat([top, pd.Series({other_label: other_n})]).sort_values(ascending=True)
colors = [tissue_domain_color(t) for t in summary.index]

fig, ax = plt.subplots(figsize=(10, 7), facecolor=PALETTE["bg"])
fig.suptitle(f"GTEx v11 Processed — Top {TOP_N} Tissues + Other",
             fontsize=14, fontweight="bold", color=PALETTE["dark"])

bars = ax.barh(range(len(summary)), summary.values,
               color=colors, edgecolor="white", linewidth=0.8, height=0.72)
ax.set_yticks(range(len(summary)))
ax.set_yticklabels(summary.index, fontsize=10)
ax.set_xlabel("Number of Samples")
ax.xaxis.grid(True, linestyle="--", alpha=0.7)
ax.set_axisbelow(True)

xmax = summary.max()
ax.set_xlim(0, xmax * 1.10)

for b in bars:
    val = int(b.get_width())
    label = f"{val:,}"
    y = b.get_y() + b.get_height() / 2
    if b.get_width() >= xmax * 0.20:
        ax.text(b.get_width() - xmax * 0.015, y, label,
                va="center", ha="right", fontsize=9,
                color="white", fontweight="bold")
    else:
        ax.text(b.get_width() + xmax * 0.012, y, label,
                va="center", ha="left", fontsize=9)

legend_handles = [
    mpatches.Patch(color=DOMAIN["neural"], label="Neural tissues"),
    mpatches.Patch(color=DOMAIN["non_neural"], label="Non-neural tissues"),
    mpatches.Patch(color=DOMAIN["other"], label="Other tissues"),
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02b_top_tissues_summary.png",
            dpi=200, bbox_inches="tight", facecolor=PALETTE["bg"])
plt.close()
print("Saved -> preprocessed/02b_top_tissues_summary.png")

# Full tissue table for reference / appendix
tissue_table = tissue_counts.reset_index()
tissue_table.columns = ["tissue", "sample_count"]
tissue_table["domain"] = tissue_table["tissue"].apply(
    lambda t: "neural" if is_neural_tissue(t) else "non_neural"
)
tissue_table.to_csv(REPO_ROOT / "results" / "tissue_sample_counts.csv", index=False)
print(f"Saved -> results/tissue_sample_counts.csv ({len(tissue_table)} tissues)")

In [18]:
# Age groups from AGE_MID (numeric) for consistency with raw EDA
meta['AGE_BIN'] = pd.cut(meta['AGE_MID'], bins=[20,30,40,50,60,70,80],
                           labels=['20-29','30-39','40-49','50-59','60-69','70-79'],
                           right=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=PALETTE["bg"])
fig.suptitle("GTEx v11 Processed — Age and Sex Breakdown",
             fontsize=14, fontweight="bold", color=PALETTE["dark"])

# SEX is now M/F directly (no mapping needed)
pivot = meta.groupby(['AGE_BIN', 'SEX']).size().unstack(fill_value=0).sort_index()
x = np.arange(len(pivot))
w = 0.35

# Grouped bar (M=purple, F=pink)
for i, (col, color) in enumerate(zip(pivot.columns,
                                      [PALETTE["primary"], PALETTE["pink"]])):
    bars = axes[0].bar(x + i*w, pivot[col], w, label=col,
                       color=color, edgecolor="white", linewidth=1.2)
axes[0].set_xticks(x + w/2)
axes[0].set_xticklabels(pivot.index, rotation=30)
axes[0].set_ylabel("Number of Samples")
axes[0].set_title("Samples per Age Group by Sex (Processed, M/F)")
axes[0].legend()
axes[0].yaxis.grid(True, linestyle="--")
axes[0].set_axisbelow(True)

# Stacked percentage
pct    = pivot.div(pivot.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(pct))
for col, color in zip(pct.columns, [PALETTE["primary"], PALETTE["pink"]]):
    axes[1].bar(x, pct[col], bottom=bottom, label=col,
                color=color, edgecolor="white", linewidth=1.2, width=0.5)
    bottom += pct[col].values
axes[1].set_xticks(x)
axes[1].set_xticklabels(pivot.index, rotation=30)
axes[1].set_ylabel("Percentage (%)")
axes[1].set_title("Sex Composition per Age Group (%) (Processed)")
axes[1].axhline(50, color=PALETTE["dark"], linestyle="--",
                linewidth=1, alpha=0.4, label="50% line")
axes[1].legend()
axes[1].yaxis.grid(True, linestyle="--")
axes[1].set_axisbelow(True)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_age_sex_breakdown.png",
            dpi=150, bbox_inches="tight", facecolor=PALETTE["bg"])
plt.close()
print("Saved -> preprocessed/03_age_sex_breakdown.png")

Saved -> preprocessed/03_age_sex_breakdown.png


In [19]:
sps = meta.groupby("SUBJID").size()

fig, ax = plt.subplots(figsize=(10, 5), facecolor=PALETTE["bg"])
fig.suptitle("GTEx v11 Processed — Tissues Collected per Subject",
             fontsize=14, fontweight="bold", color=PALETTE["dark"])

ax.hist(sps, bins=40, color=PALETTE["accent"],
        edgecolor="white", linewidth=0.8, alpha=0.9)
ax.axvline(sps.median(), color=PALETTE["primary"],
           linestyle="--", linewidth=2,
           label=f"Median: {sps.median():.0f} tissues")
ax.axvline(sps.mean(), color=PALETTE["purple"],
           linestyle=":", linewidth=2,
           label=f"Mean: {sps.mean():.1f} tissues")
ax.set_xlabel("Number of Tissues / Samples per Subject")
ax.set_ylabel("Number of Subjects")
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle="--")
ax.set_axisbelow(True)

stats_text = (f"Total subjects : {sps.shape[0]:,}\n"
              f"Min tissues    : {sps.min()}\n"
              f"Max tissues    : {sps.max()}\n"
              f"Median         : {sps.median():.0f}\n"
              f"Mean           : {sps.mean():.1f}")
ax.text(0.97, 0.95, stats_text, transform=ax.transAxes,
        fontsize=9, va="top", ha="right",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white",
                  edgecolor=PALETTE["gray"], alpha=0.9),
        family="monospace")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_samples_per_subject.png",
            dpi=150, bbox_inches="tight", facecolor=PALETTE["bg"])
plt.close()
print("Saved -> preprocessed/04_samples_per_subject.png")

Saved -> preprocessed/04_samples_per_subject.png


In [20]:
# MoE routing logic (fixed per project rules)
NEURAL_KEYWORDS = ['brain', 'spinal cord', 'nerve']
meta['domain'] = meta['SMTSD'].apply(
    lambda t: "Neural" if any(k in t.lower() for k in NEURAL_KEYWORDS)
    else "Non-Neural"
)
dc = meta['domain'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                         facecolor=PALETTE["bg"],
                         constrained_layout=True)
fig.suptitle("GTEx v11 Processed — Neural vs Non-Neural Tissue Split",
             fontsize=14, fontweight="bold", color=PALETTE["dark"])

# Donut
axes[0].pie(dc, labels=dc.index,
            colors=[DOMAIN["neural"], DOMAIN["non_neural"]],
            autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2),
            textprops={"fontsize": 11})
axes[0].set_title("Sample Proportion by Domain (Processed)")
axes[0].text(0, 0, f"{dc.sum():,}\nsamples",
             ha="center", va="center", fontsize=9, fontweight="bold")

# Age breakdown by domain (use AGE_MID bins)
meta['AGE_BIN'] = pd.cut(meta['AGE_MID'], bins=[20,30,40,50,60,70,80],
                           labels=['20-29','30-39','40-49','50-59','60-69','70-79'],
                           right=False)
pivot2 = meta.groupby(['domain', 'AGE_BIN']).size().unstack(fill_value=0).sort_index(axis=1)
age_colors = plt.cm.viridis(np.linspace(0.2, 0.9, pivot2.shape[1]))
x2 = np.arange(len(pivot2))
w2 = 0.12
for i, (col, color) in enumerate(zip(pivot2.columns, age_colors)):
    axes[1].bar(x2 + i*w2, pivot2[col], w2,
                label=col, color=color, edgecolor="white", linewidth=0.8)
axes[1].set_xticks(x2 + w2 * (len(pivot2.columns)-1) / 2)
axes[1].set_xticklabels(pivot2.index, fontsize=11)
axes[1].set_ylabel("Number of Samples")
axes[1].set_title("Age Distribution by Domain (Processed)")
axes[1].legend(title="Age Group", fontsize=8,
               title_fontsize=9, ncol=2)
axes[1].yaxis.grid(True, linestyle="--")
axes[1].set_axisbelow(True)

plt.savefig(FIGURES_DIR / "05_neural_vs_nonneural.png",
            dpi=150, bbox_inches="tight", facecolor=PALETTE["bg"])
plt.close()
print("Saved -> preprocessed/05_neural_vs_nonneural.png")

Saved -> preprocessed/05_neural_vs_nonneural.png


In [21]:
# Subject-level stats (1 row per subject, not per sample)
# Get unique subjects from processed metadata
sub_meta = meta[['SUBJID', 'SEX', 'AGE', 'DTHHRDY']].drop_duplicates(subset=['SUBJID'])

print("=" * 45)
print("  Processed Subject-level Statistics")
print("=" * 45)
print(f"Total subjects : {len(sub_meta):,}")

print("\nAge distribution (by subjects):")
print(sub_meta['AGE'].value_counts().sort_index().to_string())

print("\nSex distribution (by subjects, M/F):")
print(sub_meta['SEX'].value_counts().to_string())

print("\nHardy Scale (by subjects):")
hardy_map = {0:"Ventilator", 1:"Violent/Fast", 2:"Fast illness",
             3:"Intermediate", 4:"Slow illness"}
print(sub_meta['DTHHRDY'].map(hardy_map).value_counts().sort_index().to_string())

  Processed Subject-level Statistics
Total subjects : 930

Age distribution (by subjects):
AGE
20-29     81
30-39     73
40-49    143
50-59    299
60-69    304
70-79     30

Sex distribution (by subjects, M/F):
SEX
M    623
F    307

Hardy Scale (by subjects):
DTHHRDY
Fast illness    232
Intermediate     54
Slow illness    113
Ventilator      496
Violent/Fast     35


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5),
                          facecolor=PALETTE["bg"],
                          constrained_layout=True)
fig.suptitle("GTEx v11 Processed — Subject-level Distributions",
             fontsize=14, fontweight="bold", color=PALETTE["dark"])

# Age by subjects (use AGE from sub_meta)
age_sub = sub_meta['AGE'].value_counts().sort_index()
colors_age = plt.cm.viridis(np.linspace(0.2, 0.9, len(age_sub)))
bars = axes[0].bar(range(len(age_sub)), age_sub.values,
                   color=colors_age, edgecolor="white",
                   linewidth=1.5, width=0.7)
axes[0].set_xticks(range(len(age_sub)))
axes[0].set_xticklabels(age_sub.index, rotation=30)
axes[0].set_ylabel("Number of Subjects")
axes[0].set_title("Age Distribution\n(per subject, processed)")
axes[0].yaxis.grid(True, linestyle="--")
axes[0].set_axisbelow(True)
for b in bars:
    axes[0].text(b.get_x() + b.get_width()/2,
                 b.get_height() + 1,
                 f"{int(b.get_height()):,}",
                 ha="center", fontsize=9)

# Sex by subjects
sex_sub = sub_meta["SEX"].replace({"M": "Male", "F": "Female"}).value_counts()
wedges, texts, autotexts = axes[1].pie(
    sex_sub,
    labels=sex_sub.index,
    colors=[PALETTE["primary"], PALETTE["pink"]],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2),
    textprops={"fontsize": 11}
)
for at in autotexts:
    at.set_fontweight("bold")
    at.set_color("white")
axes[1].set_title("Sex Distribution\n(per subject, processed)")
axes[1].text(0, 0, f"{sex_sub.sum():,}\nsubjects",
             ha="center", va="center",
             fontsize=9, fontweight="bold")

plt.savefig(FIGURES_DIR / "06_subject_distributions.png",
            dpi=150, bbox_inches="tight", facecolor=PALETTE["bg"])
plt.close()
print("Saved -> preprocessed/06_subject_distributions.png")

Saved -> preprocessed/06_subject_distributions.png
